## Exploratory Data Analysis 
Explore preprocessing options with raw data

In [ ]:
from pathlib import Path
import sys

# Create PROJECT_ROOT and add it to sys.path so Python can search for customized module
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np
from backend.services.openfigi_client import populate_security, fill_missing_trade_symbols

In [3]:
# File Location
Y24_25_file = '../data/2024_2025.csv'
Y25_26_file = '../data/2025_2026.csv'

In [4]:
Y24_25 = pd.read_csv(Y24_25_file)
Y25_26 = pd.read_csv(Y25_26_file)

# Drop comments at the bottom of the files
Y24_25_clean = Y24_25.dropna(subset=['Account'])
Y25_26_clean = Y25_26.dropna(subset=['Account'])

data = pd.concat([Y24_25_clean, Y25_26_clean], ignore_index=True).sort_values(by='Run Date', ascending=False)
data.columns

Index(['Run Date', 'Account', 'Account Number', 'Action', 'Symbol',
       'Description', 'Type', 'Price ($)', 'Quantity', 'Commission ($)',
       'Fees ($)', 'Accrued Interest ($)', 'Amount ($)', 'Settlement Date'],
      dtype='str')

### Transaction Classification and Validation

This section classifies each account activity using two transaction dimensions and one security dimension:

* `transaction_type` - broad transaction category
* `transaction_subtype` - specific transaction action
* `security_type` - type of security involved in the transaction

The main transaction types are:

* `corporate_action` - non-trade events that modify a security position

  * `reverse_split` - reduces the number of shares and increases the price and cost basis per share
  * `split` - increases the number of shares and reduces the price and cost basis per share

* `trade` - security purchases and sales

  * `sold` - securities sold
  * `bought` - securities purchased

* `distribution` - investment income and related reinvestments

  * `ordinary_dividend` - dividend distribution received from a stock, ETF, mutual fund, or money-market fund
  * `long_term_cap_gain` - long-term capital-gain distribution paid by a fund
  * `reinvestment` - distribution used to purchase additional shares

* `transfer` - movement of cash or assets between accounts

  * `internal_account_transfer` - transfer between investment accounts, such as brokerage-to-crypto or brokerage-to-brokerage
  * `electronic_funds_transfer` - transfer between an investment account and an external bank account
  * `other_transfer` - transfer that does not match a known transfer method

* `expense` - charges deducted from the account

  * `fee` - account or transaction fee charged
  * `foreign_tax` - foreign tax withheld from investment income

* `other` - actions that do not match the current classification rules

Security metadata is stored separately from transaction classification. Each resolved security includes:

* `security_name` - full security name
* `security_type` - normalized asset category
* `security_type_raw` - original security type returned by the metadata source
* `security_source` - source used to resolve the security metadata

The main security types include:

* `common_stock`
* `preferred_stock`
* `depositary_receipt`
* `etf`
* `mutual_fund`
* `money_market_fund`
* `closed_end_fund`
* `unit_investment_trust`
* `crypto`

The sign of `amount` represents cash movement:

* positive values represent cash entering the account
* negative values represent cash leaving the account

Expected sign rules include:

* `sold` - positive
* `bought` - negative
* `ordinary_dividend` - positive
* `long_term_cap_gain` - positive
* `reinvestment` - negative
* `fee` - negative
* `foreign_tax` - negative

Transfers may be positive or negative depending on direction. Corporate actions may have zero amounts or related cash adjustments and therefore do not follow a fixed sign rule.

Corporate actions are excluded from ordinary trade counts but retained for holdings and cost-basis calculations. A split or reverse split changes share quantity and per-share cost basis while generally preserving total cost basis.

For the current Fidelity dataset, ordinary dividends and long-term capital-gain distributions from `mutual_fund` and `money_market_fund` securities are expected to have corresponding negative `reinvestment` transactions.

These distribution-and-reinvestment groups should be matched using:

* `account_number`
* `symbol`
* `run_date`

Each matched group should:

* contain at least one `ordinary_dividend` or `long_term_cap_gain`
* contain at least one `reinvestment`
* have a net `amount` approximately equal to zero

A numerical tolerance should be used because floating-point calculations may produce very small residual differences.

Stock and ETF dividends are not assumed to be reinvested automatically. Their treatment depends on the actual transactions recorded in the account.

Any unresolved security metadata is omitted from the completed security lookup table and identified using the difference between the requested symbols and the resolved security symbols. These securities can then be handled by a separate fallback process.

In [5]:
# Organize each row by transaction type and subtype
action = data["Action"].fillna("")
description = data["Description"].fillna("")

# Trade actions
is_sold = action.str.contains(r"\bsold\b", case=False)
is_bought = action.str.contains(r"\bbought\b", case=False)
is_trade = is_sold | is_bought

# Corporate actions
is_reverse_split = description.str.contains(
    r"\br\s*/\s*s\b|\breverse\s+split\b",
    case=False,
)

# Exclude reverse splits from the regular split condition
is_split = (
    description.str.contains(r"\bsplit\b", case=False)
    & ~is_reverse_split
)

is_corporate_action = is_reverse_split | is_split

# Distribution actions
# Security type will later distinguish stock, ETF, mutual-fund,
# and money-market dividends.
is_reinvestment = action.str.contains(r"\breinvestment\b", case=False)
is_ordinary_dividend = action.str.contains(r"\bdividend\b", case=False)

is_long_term_cap_gain = action.str.contains(
    r"\blong[-\s]+term\s+cap(?:ital)?\s+gain\b",
    case=False,
)

is_distribution = (
    is_reinvestment
    | is_ordinary_dividend
    | is_long_term_cap_gain
)

# Transfer actions
is_transfer = action.str.contains(r"\btransfer\b", case=False)

is_electronic_funds_transfer = action.str.contains(
    r"\b(?:electronic funds transfer|eft)\b",
    case=False,
)

is_internal_account_transfer = (
    is_transfer
    & action.str.contains(r"\b(?:brokerage|crypto)\b", case=False)
)

# Expense actions
is_fee = action.str.contains(r"\bfee charged\b", case=False)
is_foreign_tax = action.str.contains(r"\bforeign tax paid\b", case=False)
is_expense = is_fee | is_foreign_tax

# Assign the broad transaction category
data["transaction_type"] = np.select(
    [
        is_corporate_action,
        is_trade,
        is_distribution,
        is_transfer,
        is_expense,
    ],
    [
        "corporate_action",
        "trade",
        "distribution",
        "transfer",
        "expense",
    ],
    default="other",
)

# Assign the specific transaction action
# More specific conditions must appear before broader conditions.
data["transaction_subtype"] = np.select(
    [
        is_reverse_split,
        is_split,
        is_sold,
        is_bought,
        is_reinvestment,
        is_ordinary_dividend,
        is_long_term_cap_gain,
        is_electronic_funds_transfer,
        is_internal_account_transfer,
        is_transfer,
        is_fee,
        is_foreign_tax,
    ],
    [
        "reverse_split",
        "split",
        "sold",
        "bought",
        "reinvestment",
        "ordinary_dividend",
        "long_term_cap_gain",
        "electronic_funds_transfer",
        "internal_account_transfer",
        "other_transfer",
        "fee",
        "foreign_tax",
    ],
    default="other",
)

In [6]:
# Date validation and rename
data['Run Date'] = pd.to_datetime(data['Run Date'], format='%m/%d/%Y', errors='coerce')
data['Settlement Date'] = pd.to_datetime(data['Settlement Date'], format='%m/%d/%Y', errors='coerce')
data.rename(columns={'Run Date': 'run_date', 'Settlement Date': 'settlement_date'}, inplace=True)

In [7]:
# Check numeric columns and rename
naming_dict = {
    'Price ($)': 'price',
    'Quantity': 'quantity',
    'Commission ($)': 'commission',
    'Fees ($)': 'fees',
    'Accrued Interest ($)': 'accrued_interest',
    'Amount ($)': 'amount',
    'Account': 'account',
    'Account Number': 'account_number',
    'Action': 'action',
    'Symbol': 'symbol',
    'Description': 'description',
    'Type': 'type'
}
numeric_columns = ['Price ($)', 'Quantity', 'Commission ($)', 'Fees ($)', 'Accrued Interest ($)', 'Amount ($)']

for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors='coerce')

data = data.rename(columns=naming_dict)

In [8]:
# Handle NULL symbol value
unresolved_data = fill_missing_trade_symbols(data)

In [9]:
# Building security map
security_map = populate_security(
    data["symbol"].dropna().unique().tolist()
)

In [10]:
original = set(data["symbol"].dropna().unique().tolist())
security_map_list = set(security_map.keys())
result = original - security_map_list
result

set()

In [11]:
# Convert the nested security dictionary into a lookup table
security_table = pd.DataFrame.from_dict(
    security_map,
    orient="index",
)

security_table.index.name = "symbol"

# Map security metadata onto each transaction
data = data.join(
    security_table,
    on="symbol",
    how="left",
    validate="many_to_one",
)

In [12]:
# Resolve primary key conflict
def pk_checker(df: pd.DataFrame):
    """
    Detect repeated transaction keys.

    Fidelity does not provide timestamps or transaction IDs, so legitimate
    trades may share the same date, symbol, price, quantity, and amount.

    Current rule:
    - Keep repeated rows when the ticker's total quantity is >= 0.
    - If total quantity is negative, remove duplicated sell rows until the
      quantity is no longer negative.
    """

    pk = [
        "run_date", "account_number", "symbol", "price",
        "quantity", "amount", "settlement_date",
        "transaction_subtype"
    ]

    # Work on a copy so the original DataFrame is preserved
    cleaned_df = df.copy()

    # Rows that repeat a proposed transaction key
    duplicate = cleaned_df[
        cleaned_df.duplicated(subset=pk, keep="first")
    ].copy()

    if duplicate.empty:
        print("No repeated transaction keys found.")
        return cleaned_df, duplicate

    removed_index = []

    # Only inspect tickers involved in repeated keys
    checklist = duplicate["symbol"].dropna().unique()

    for tick in checklist:
        subset = cleaned_df[
            cleaned_df["symbol"].eq(tick)
        ].sort_values(by=["run_date", "settlement_date"])

        checker = 0

        for row in subset.itertuples():
            if pd.notna(row.quantity):
                checker += row.quantity
        
        # Adjust for float precision
        tolerance = 1e-9

        if abs(checker) < tolerance:
            checker = 0

        print(f"{tick}: total quantity = {checker}")

        # A zero balance is valid; only a negative balance triggers removal
        if checker < 0:

            # Only duplicated sell rows could correct a negative balance
            candidates = duplicate[
                duplicate["symbol"].eq(tick)
                & duplicate["quantity"].lt(0)
            ].sort_values(
                by=["run_date", "settlement_date"],
                ascending=False,
            )

            for index, row in candidates.iterrows():
                removed_index.append(index)

                # Subtracting a negative removed quantity raises the balance
                checker -= row["quantity"]

                print(
                    f"Removing index {index}: "
                    f"{row['quantity']} shares; "
                    f"new quantity = {checker}"
                )

                if checker >= 0:
                    break
            
            if checker < 0:
                print(f"{tick}: still negative after checking duplicates.")
        else:
            print(f"{tick}: repeated rows retained.")

    removed_rows = cleaned_df.loc[removed_index].copy()
    cleaned_df = cleaned_df.drop(index=removed_index)

    return cleaned_df, removed_rows

In [ ]:
def check_sign_consistency(
    df: pd.DataFrame,
    tolerance: float = 1e-9,
) -> pd.DataFrame:
    """
    Return transactions whose amount sign conflicts with their subtype.
    """

    expected_positive = {
        "sold",
        "ordinary_dividend",
        "long_term_cap_gain",
    }

    expected_negative = {
        "bought",
        "reinvestment",
        "fee",
        "foreign_tax",
    }

    positive_errors = (
        df["transaction_subtype"].isin(expected_positive)
        & (df["amount"] <= tolerance)
    )

    negative_errors = (
        df["transaction_subtype"].isin(expected_negative)
        & (df["amount"] >= -tolerance)
    )

    missing_amount = (
        df["transaction_subtype"].isin(
            expected_positive | expected_negative
        )
        & df["amount"].isna()
    )

    errors = df[
        positive_errors
        | negative_errors
        | missing_amount
    ].copy()

    return errors

sign_errors = check_sign_consistency(data)

if sign_errors.empty:
    print("All transaction signs are valid.")
else:
    print(f"Invalid transaction signs: {len(sign_errors)}")

All transaction signs are valid.


In [14]:
tolerance = 1e-9

# Fund distributions that are expected to be reinvested
fund_types = {
    "mutual_fund",
    "money_market_fund",
}

fund_distribution_subtypes = {
    "ordinary_dividend",
    "long_term_cap_gain",
    "reinvestment",
}

fund_distributions = data[
    data["security_type"].isin(fund_types)
    & data["transaction_subtype"].isin(fund_distribution_subtypes)
].copy()

# Distribution and reinvestment rows should net to zero
distribution_check = (
    fund_distributions
    .groupby(
        [
            "account_number",
            "symbol",
            "run_date",
        ],
        dropna=False,
    )
    .agg(
        net_amount=("amount", "sum"),
        transaction_count=("amount", "size"),
        subtypes=(
            "transaction_subtype",
            lambda values: sorted(set(values)),
        ),
    )
    .reset_index()
)

incorrect_distributions = distribution_check[
    distribution_check["net_amount"].abs() > tolerance
]

if incorrect_distributions.empty:
    print("All fund distributions were reinvested correctly.")
else:
    print(
        "Incorrect or unmatched fund distributions:",
        len(incorrect_distributions),
    )
    display(incorrect_distributions)

All fund distributions were reinvested correctly.
